# Statistical baselines — evaluation protocol and three models

Implements roadmap §3 (metrics / rolling-origin protocol) and §4 (seasonal naive,
climatology, ARIMA + Fourier). The goal is **not** a baselines paper — it is a
reference table the multimodal model must beat.

**Artifacts this notebook must leave behind**

1. `src/.../metrics/evaluate.py` — scoring function reusable by later models
2. `results/baseline_scores.parquet` — 3 models × 2 horizons (MAE-log + skill)
3. The article table printed at the end

## Why this protocol

| Choice | Why |
|---|---|
| **Point forecasts + MAE on `log1p`** | Config decision: distributions/CRPS are out of scope for Phase 1. Log scale stops SP/MG from owning the national average (EDA overdispersion). |
| **Skill vs seasonal naive** | Absolute MAE is hard to read across horizons; % improvement vs a dumb seasonal rule is the honest headline. |
| **Rolling origin, 6-year window** | Mimics operational updating without leaking future data; fixed window avoids giving later origins an unfair longer history. |
| **Horizons 4 and 12 only** | 4 weeks = operational; 12 weeks = the multimodal target in `config.yml`. More horizons would inflate the table without changing the story. |
| **Held-out = last 2 dengue seasons (EW41→EW40)** | Out-of-sample block that matches how seasons are discussed in the Brazilian literature — not a random 20% slice. |

Dropped on purpose: MAPE, RMSE, coverage, per-UF appendix tables.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
import yaml

from arboviruses_series_forecasting.metrics import score_forecasts
from arboviruses_series_forecasting.models import run_baseline_forecasts

REPO_ROOT = Path("..").resolve()
config = yaml.safe_load((REPO_ROOT / "config.yml").read_text())

PROCESSED_PATH = REPO_ROOT / config["paths"]["processed"] / "dengue_uf_ew.parquet"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SCORES_PATH = RESULTS_DIR / "baseline_scores.parquet"
FORECASTS_PATH = RESULTS_DIR / "baseline_forecasts.parquet"

eval_cfg = config["evaluation"]
agg_cfg = config["aggregation"]
HORIZONS = list(eval_cfg["horizons"])
TRAIN_YEARS = int(eval_cfg["training_window_years"])
HELD_OUT_SEASONS = int(eval_cfg["held_out_seasons"])
SEASON_START = int(agg_cfg["season_boundary"]["start"])
SEASON_END = int(agg_cfg["season_boundary"]["end"])

assert PROCESSED_PATH.exists(), f"Run the EDA notebook first: missing {PROCESSED_PATH}"
print(f"panel:    {PROCESSED_PATH}")
print(f"horizons: {HORIZONS}")
print(f"train:    {TRAIN_YEARS}y rolling window")
print(f"held-out: last {HELD_OUT_SEASONS} seasons (EW{SEASON_START}→EW{SEASON_END})")

panel:    /home/caioolubini/Projects/arboviruses-series-forecasting/data/processed/dengue_uf_ew.parquet
horizons: [4, 12]
train:    6y rolling window
held-out: last 2 seasons (EW41→EW40)


## Load the forecasting panel

Same UF × EW probable-case series built in `01_eda.ipynb`. Baselines see only this
unimodal target — no climate, Trends, or news (that separation *is* the Phase 2 claim).

In [2]:
dengue = pd.read_parquet(PROCESSED_PATH)
dengue["week_start"] = pd.to_datetime(dengue["week_start"])
dengue[["uf", "week_start", "cases", "ew_year", "ew"]].head()

,uf,week_start,cases,ew_year,ew
0,AC,2010-01-03,759,2010,1
1,AC,2010-01-10,895,2010,2
2,AC,2010-01-17,889,2010,3
3,AC,2010-01-24,1231,2010,4
4,AC,2010-01-31,1809,2010,5


## Why these three baselines

| Model | Role in the paper | Why it earns a slot |
|---|---|---|
| **Seasonal naive** (`ŷ_{t+h} = y_{t+h-52}`) | Skill-score denominator | If you cannot beat “same week last year”, you have not earned a Methods section. |
| **Climatology** (training-window median by EW) | Strong long-horizon null | Ignores recent shocks; often hard to beat at 12 weeks when epidemics are irregular. |
| **ARIMA + Fourier on `log1p`** | Literature-standard statistical model | Fourier harmonics replace SARIMA: seasonal period `m = 52` is numerically miserable. K and a small `(p,d,q)` grid are chosen by AICc once per UF. |

Not included (roadmap cuts): ETS, NB-GLM, VAR, SARIMAX with covariates. Six models is a
baselines *paper*; three is a baselines *section*.

## Run rolling-origin forecasts

For each UF, each horizon, and each held-out target week:

1. Origin = target − horizon (information available when the forecast is issued)
2. Train on the previous 6×52 weeks through the origin
3. Emit a point forecast on the **count** scale

ARIMA order / Fourier K are selected once per UF on the earliest training window, then
coefficients are refit at every origin (orders stay fixed so the run is tractable).

In [3]:
forecasts, arima_specs, held_seasons = run_baseline_forecasts(
    dengue,
    horizons=HORIZONS,
    training_window_years=TRAIN_YEARS,
    held_out_seasons=HELD_OUT_SEASONS,
    season_start=SEASON_START,
    season_end=SEASON_END,
)

forecasts.to_parquet(FORECASTS_PATH, index=False)

spec_table = pd.DataFrame(
    {
        "uf": list(arima_specs),
        "order": [str(arima_specs[uf].order) for uf in arima_specs],
        "fourier_k": [arima_specs[uf].fourier_k for uf in arima_specs],
    }
).sort_values("uf")

print(f"held-out seasons (end year): {held_seasons}")
print(f"forecast rows: {len(forecasts):,}")
print(f"saved forecasts → {FORECASTS_PATH.relative_to(REPO_ROOT)}")
spec_table.head(10)

held-out seasons (end year): [2024, 2025]
forecast rows: 16,848
saved forecasts → results/baseline_forecasts.parquet


,uf,order,fourier_k
0,AC,"(1, 1, 0)",2
1,AL,"(1, 1, 1)",1
2,AM,"(1, 1, 0)",1
3,AP,"(1, 1, 1)",2
4,BA,"(1, 1, 0)",1
5,CE,"(1, 1, 0)",1
6,DF,"(1, 1, 1)",3
7,ES,"(1, 1, 0)",1
8,GO,"(1, 1, 1)",2
9,MA,"(1, 1, 1)",3


## Score with the shared harness

`score_forecasts(forecasts, truth)` is the Phase 1 deliverable the multimodal model must
call unchanged. Headline = MAE on `log1p`; skill = % improvement vs seasonal naive.

In [4]:
truth = dengue[["uf", "week_start", "cases"]]
scores = score_forecasts(forecasts, truth, skill_baseline_model="seasonal_naive")
scores.to_parquet(SCORES_PATH, index=False)

print(f"saved scores → {SCORES_PATH.relative_to(REPO_ROOT)}")
scores

saved scores → results/baseline_scores.parquet


,model,horizon,mae_log1p,n,skill_vs_seasonal_naive
0,arima_fourier,4,0.375606,2808,59.719240
1,climatology,4,1.006449,2808,-7.933695
2,seasonal_naive,4,0.932470,2808,0.000000
3,arima_fourier,12,0.656535,2808,29.591766
4,climatology,12,1.006449,2808,-7.933695
5,seasonal_naive,12,0.932470,2808,0.000000


## Article table — 3 baselines × 2 horizons

One number readers will remember: skill of the best model vs seasonal naive at 4 weeks,
and how that skill degrades at 12 weeks.

In [5]:
table = scores.pivot(index="model", columns="horizon", values=["mae_log1p", "skill_vs_seasonal_naive"])
table.columns = [f"h{h}_{metric}" for metric, h in table.columns]
table = table.reindex(["seasonal_naive", "climatology", "arima_fourier"])

best = (
    scores.loc[scores["model"] != "seasonal_naive"]
    .sort_values("skill_vs_seasonal_naive", ascending=False)
    .groupby("horizon", as_index=False)
    .first()
)

for _, row in best.iterrows():
    print(
        f"Best at h={int(row['horizon'])}: {row['model']} "
        f"({row['skill_vs_seasonal_naive']:+.1f}% vs seasonal naive, "
        f"MAE-log={row['mae_log1p']:.3f})"
    )

table.round(3)

Best at h=4: arima_fourier (+59.7% vs seasonal naive, MAE-log=0.376)
Best at h=12: arima_fourier (+29.6% vs seasonal naive, MAE-log=0.657)


,h4_mae_log1p,h12_mae_log1p,h4_skill_vs_seasonal_naive,h12_skill_vs_seasonal_naive
model,,,,
seasonal_naive,0.932,0.932,0.000,0.000
climatology,1.006,1.006,-7.934,-7.934
arima_fourier,0.376,0.657,59.719,29.592


## Methods paragraph (draft)

> We evaluated three unimodal statistical baselines on probable dengue counts at UF ×
> epidemiological-week resolution: seasonal naive (lag 52), EW-specific climatological
> medians, and ARIMA with Fourier harmonics fitted on `log1p` counts (harmonic order and a
> small ARIMA order grid selected by AICc). Forecasts used a rolling origin with a fixed
> six-year training window, horizons of 4 and 12 weeks, and a held-out block of the two most
> recent dengue seasons (EW41–EW40). Accuracy was summarised as MAE on the `log1p` scale and
> as percent improvement relative to seasonal naive, averaged across UFs and held-out weeks.

Next phase: call `score_forecasts` on the multimodal model with the same truth panel.